# 04 · Sentiment Analysis

**Purpose:** Attach a sentiment signal to every (article, ticker) pair.

- **Primary source:** LLM scores already stored in `sentiment_scores` (PostgreSQL).
- **Fallback:** Simple French lexicon for articles the LLM never processed.

**Input:** `articles_mapped.parquet`, `sentiments.parquet`  
**Output:** `articles_enriched.parquet`  
New columns: `sentiment`, `score`, `confidence`, `sentiment_source`

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

NB_DIR = Path().resolve()
sys.path.insert(0, str(NB_DIR))
from utils import load, save

In [ ]:
articles_mapped = load("articles_mapped")
sentiments      = load("sentiments")

print(f"Mapped (article, ticker) pairs: {len(articles_mapped):,}")
print(f"LLM sentiment scores:           {len(sentiments):,}")

## 1 · Merge LLM sentiments

In [ ]:
enriched = articles_mapped.merge(
    sentiments[["article_url", "sentiment", "score", "confidence"]],
    on="article_url",
    how="left",
)

llm_coverage = enriched["sentiment"].notna().mean()
print(f"LLM coverage: {llm_coverage:.1%} of (article, ticker) pairs have a score")

enriched["sentiment_source"] = enriched["sentiment"].notna().map(
    {True: "llm", False: "lexicon"}
)

## 2 · French lexicon fallback

In [ ]:
POSITIVE_STEMS = [
    "croissance", "hausse", "progression", "bénéfice", "profit",
    "succès", "record", "expansion", "rebond", "gain", "dynamique",
    "excellent", "performant", "amélioration", "solide", "fort",
    "optimiste", "positif", "accroissement", "développement",
]

NEGATIVE_STEMS = [
    "baisse", "chute", "perte", "recul", "déficit", "déclin",
    "difficile", "crise", "risque", "incertitude", "ralentissement",
    "préoccupation", "tension", "négatif", "détérioration",
    "pessimiste", "fragilité", "dégradation",
]


def lexicon_sentiment(text: str) -> dict:
    words = text.lower().split()
    pos = sum(1 for w in words if any(s in w for s in POSITIVE_STEMS))
    neg = sum(1 for w in words if any(s in w for s in NEGATIVE_STEMS))
    total = pos + neg
    if total == 0:
        return {"sentiment": "neutral", "score": 0.0, "confidence": 0.0}
    score = (pos - neg) / total
    label = "positive" if score > 0.1 else "negative" if score < -0.1 else "neutral"
    conf  = round(min(total / max(len(words), 1) * 5, 1.0), 3)  # proxy for confidence
    return {"sentiment": label, "score": round(score, 3), "confidence": conf}

In [ ]:
no_score = enriched["sentiment"].isna()
print(f"Rows needing lexicon fallback: {no_score.sum():,}")

if no_score.any():
    lex_results = enriched.loc[no_score, "text"].apply(lexicon_sentiment).apply(pd.Series)
    enriched.loc[no_score, "sentiment"]  = lex_results["sentiment"].values
    enriched.loc[no_score, "score"]      = lex_results["score"].values
    enriched.loc[no_score, "confidence"] = lex_results["confidence"].values

print("\nSentiment distribution:")
print(enriched["sentiment"].value_counts().to_string())

## 3 · Quality check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

enriched["sentiment"].value_counts().plot.bar(ax=axes[0], color=["#2196F3", "#4CAF50", "#F44336"])
axes[0].set_title("Sentiment label distribution")
axes[0].set_xlabel("")

enriched["score"].hist(bins=40, ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Sentiment score distribution")
axes[1].set_xlabel("Score (-1 = very negative, +1 = very positive)")

plt.tight_layout()
plt.show()

print("\nScore stats by label:")
print(enriched.groupby("sentiment")["score"].describe().round(3).to_string())

In [ ]:
save(enriched, "articles_enriched")